# Example 1: tracing a potential unit-annotation error to its source

CAPRICHO preserves flagged measurements and the identifiers needed to inspect them. This notebook follows two flagged pairs from the Case Study 1 output through ChEMBL metadata to the original publications and Supporting Information.

A difference of exactly three log units is a **potential** annotation error, not proof of one. The examples show two possible outcomes of source verification: one value differs from the source table, whereas another apparently unusual difference is confirmed by the source. This notebook only documents those observations; it does not alter the ChEMBL database or prescribe how a user should act on them.

In [1]:
from pathlib import Path
import sqlite3

import pandas as pd

CHEMBL_VERSION = 32
ACTIVITY_IDS = [1512257, 1512264, 20679701, 20679711]

if Path.cwd().name == "notebooks":
    NOTEBOOK_DIR = Path.cwd()
elif (Path.cwd() / "notebooks").is_dir():
    NOTEBOOK_DIR = Path.cwd() / "notebooks"
else:
    raise RuntimeError("Run this notebook from the repository root or notebooks directory.")

case1_path = (
    NOTEBOOK_DIR
    / "data"
    / "max-curated"
    / f"curated-IC50-NoAssayOverlap-chembl{CHEMBL_VERSION}_not_aggregated.csv"
)

trace_columns = [
    "activity_id",
    "molecule_chembl_id",
    "assay_chembl_id",
    "target_chembl_id",
    "document_chembl_id",
    "doi",
    "standard_type",
    "standard_value",
    "standard_units",
    "pchembl_value",
    "data_dropping_comment",
]

case1_trace = pd.read_csv(case1_path, usecols=trace_columns, low_memory=False)
case1_trace = (
    case1_trace.loc[case1_trace["activity_id"].isin(ACTIVITY_IDS), trace_columns]
    .sort_values("activity_id")
    .reset_index(drop=True)
)
assert set(case1_trace["activity_id"]) == set(ACTIVITY_IDS)
case1_trace

,activity_id,molecule_chembl_id,assay_chembl_id,target_chembl_id,document_chembl_id,doi,standard_type,standard_value,standard_units,pchembl_value,data_dropping_comment
0,1512257,CHEMBL195218,CHEMBL829589,CHEMBL279,CHEMBL1144449,10.1021/jm0501275,IC50,62.000,nM,7.21,Unit Annotation Error & Insufficient assay ove...
1,1512264,CHEMBL195218,CHEMBL829507,CHEMBL279,CHEMBL1144449,10.1021/jm0501275,IC50,0.062,nM,10.21,Unit Annotation Error & Assay size < 20
2,20679701,CHEMBL4640769,CHEMBL4622499,CHEMBL1163125,CHEMBL4619816,10.1021/acsmedchemlett.0c00247,IC50,10000.000,nM,5.00,Unit Annotation Error & Mutation keyword in as...
3,20679711,CHEMBL4640769,CHEMBL4622498,CHEMBL1163125,CHEMBL4619816,10.1021/acsmedchemlett.0c00247,IC50,10.000,nM,8.00,Unit Annotation Error & Mutation keyword in as...


## Add the source record identifiers

The CAPRICHO output already retains the activity, molecule, assay, target, document, and DOI fields. The short query below adds ChEMBL's source compound key and source record ID for these four activities.

The local ChEMBL SQLite file is opened with `mode=ro` (read-only). No values are written back to the downloaded database.

In [2]:
db_path = Path.home() / ".data" / "chembl" / str(CHEMBL_VERSION) / f"chembl_{CHEMBL_VERSION}.db"
if not db_path.exists():
    raise FileNotFoundError(f"Local ChEMBL {CHEMBL_VERSION} database not found: {db_path}")

placeholders = ",".join("?" for _ in ACTIVITY_IDS)
query = f"""
    SELECT
        act.activity_id,
        md.chembl_id AS molecule_chembl_id,
        ass.chembl_id AS assay_chembl_id,
        td.chembl_id AS target_chembl_id,
        docs.chembl_id AS document_chembl_id,
        records.compound_key AS source_compound_key,
        records.record_id AS source_record_id,
        act.type AS source_type,
        act.value AS source_value,
        act.units AS source_units,
        act.standard_value,
        act.standard_units,
        act.pchembl_value,
        docs.doi
    FROM activities AS act
    JOIN assays AS ass ON ass.assay_id = act.assay_id
    JOIN target_dictionary AS td ON td.tid = ass.tid
    JOIN docs ON docs.doc_id = ass.doc_id
    JOIN molecule_dictionary AS md ON md.molregno = act.molregno
    JOIN compound_records AS records ON records.record_id = act.record_id
    WHERE act.activity_id IN ({placeholders})
    ORDER BY act.activity_id
"""

database_uri = f"{db_path.resolve().as_uri()}?mode=ro"
with sqlite3.connect(database_uri, uri=True) as connection:
    chembl_trace = pd.read_sql_query(query, connection, params=ACTIVITY_IDS)

chembl_trace

,activity_id,molecule_chembl_id,assay_chembl_id,target_chembl_id,document_chembl_id,source_compound_key,source_record_id,source_type,source_value,source_units,standard_value,standard_units,pchembl_value,doi
0,1512257,CHEMBL195218,CHEMBL829589,CHEMBL279,CHEMBL1144449,24,391924,IC50,62.000,nM,62.000,nM,7.21,10.1021/jm0501275
1,1512264,CHEMBL195218,CHEMBL829507,CHEMBL279,CHEMBL1144449,24,391924,IC50,0.062,nM,0.062,nM,10.21,10.1021/jm0501275
2,20679701,CHEMBL4640769,CHEMBL4622499,CHEMBL1163125,CHEMBL4619816,30,3482622,pIC50,5.000,None,10000.000,nM,5.00,10.1021/acsmedchemlett.0c00247
3,20679711,CHEMBL4640769,CHEMBL4622498,CHEMBL1163125,CHEMBL4619816,30,3482622,pIC50,8.000,None,10.000,nM,8.00,10.1021/acsmedchemlett.0c00247


## Verify the values in the source material

The DOI identifies the publication, and ChEMBL's document record links the activities to it. The corresponding Supporting Information can be inspected at the public links below.

- **VEGFR-2 example:** [publication](https://doi.org/10.1021/jm0501275) · [Supporting Information](https://ndownloader.figshare.com/files/5119336). Section II, “Table of Pertinent Data for C6-Carbamate Analogues” (pp. S2–S3), reports a VEGFR-2 IC50 of **62 nM** for the `(CH2)3SO2CH3` analogue.
- **BRD4 example:** [publication](https://doi.org/10.1021/acsmedchemlett.0c00247) · [Supporting Information](https://ndownloader.figshare.com/files/23759241). Table S2 (p. S11) reports pIC50 values of **5.0 for BRD4 BD1** and **8.0 for BRD4 BD2** for compound 30, together with the stated 1000-fold selectivity.

The concise comparison below records what was observed in ChEMBL and in the source tables. It does not apply any correction.

In [3]:
source_comparison = pd.DataFrame(
    [
        {
            "activity_id": 1512257,
            "source_location": "jm0501275 SI, section II, pp. S2-S3",
            "chembl_value": "62 nM",
            "source_value": "62 nM",
            "observation": "agrees with source",
        },
        {
            "activity_id": 1512264,
            "source_location": "jm0501275 SI, section II, pp. S2-S3",
            "chembl_value": "0.062 nM",
            "source_value": "62 nM",
            "observation": "differs from source table",
        },
        {
            "activity_id": 20679701,
            "source_location": "0c00247 SI, Table S2, p. S11",
            "chembl_value": "pIC50 5.0",
            "source_value": "pIC50 5.0",
            "observation": "agrees with source",
        },
        {
            "activity_id": 20679711,
            "source_location": "0c00247 SI, Table S2, p. S11",
            "chembl_value": "pIC50 8.0",
            "source_value": "pIC50 8.0",
            "observation": "agrees with source",
        },
    ]
)
source_comparison

,activity_id,source_location,chembl_value,source_value,observation
0,1512257,"jm0501275 SI, section II, pp. S2-S3",62 nM,62 nM,agrees with source
1,1512264,"jm0501275 SI, section II, pp. S2-S3",0.062 nM,62 nM,differs from source table
2,20679701,"0c00247 SI, Table S2, p. S11",pIC50 5.0,pIC50 5.0,agrees with source
3,20679711,"0c00247 SI, Table S2, p. S11",pIC50 8.0,pIC50 8.0,agrees with source


## Take-away

CAPRICHO's flag provides a starting point for source verification while preserving the relevant ChEMBL identifiers. In the VEGFR-2 example, both activities map to the same molecule, target, document, source compound key, and source record, but one reported value differs from the Supporting Information. In the BRD4 example, the source confirms that the three-log-unit difference reflects the reported assay result rather than a unit annotation problem.

This distinction is why potential unit-annotation-error flags should support inspection rather than automatic deletion. The notebook stops at source verification and leaves the original ChEMBL database unchanged.